### Passo 1: Importações

In [6]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Case").getOrCreate()

from pyspark.sql import functions as F
from pyspark.sql.functions import date_format
import pandas as pd
from pyspark.sql.window import Window

### Passo 2: Leitura das bases

In [16]:
PATH = '/home/tatiane/repositorios/data-projects/02_data_analysis/etl_banco_digital/bronze/'

agencias = spark.read.csv(f"{PATH}agencias.csv", header=True)
clientes = spark.read.csv(f"{PATH}clientes.csv", header=True)
pix = spark.read.csv(f"{PATH}pix.csv", header=True)
contas = spark.read.csv(f"{PATH}contas.csv", header=True)
transacoes = spark.read.csv(f"{PATH}transacoes_cartao.csv", header=True)

### Passo 3: Tratamento das bases

#### Clientes

In [40]:
print('Qtd. registros:', clientes.count(), '| Quantidade de colunas: ', len(clientes.columns))
clientes.printSchema()

Qtd. registros: 10020 | Quantidade de colunas:  7
root
 |-- id_cliente: string (nullable = true)
 |-- sexo: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- renda: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- data_nascimento: string (nullable = true)



- Padronizando os tipos de variáveis

In [41]:
clientes = clientes.withColumn("renda", F.col("renda").cast("float"))
clientes = clientes.withColumn("data_nascimento_formatada", date_format("data_nascimento", "dd/MM/yyyy"))

In [43]:
clientes.show(3)

+----------+----+---+-------+------------+-------------+-------------------+-------------------------+
|id_cliente|sexo| uf|  renda|      cidade|         nome|    data_nascimento|data_nascimento_formatada|
+----------+----+---+-------+------------+-------------+-------------------+-------------------------+
|         1|   F| RJ|17706.0|     Niterói|Cliente_00001|1956-07-10 19:27:27|               10/07/1956|
|         2|   M| RS|13387.0|Porto Alegre|Cliente_00002|1990-03-06 20:25:22|               06/03/1990|
|         3|   F| SC|16528.0|   Joinville|Cliente_00003|1974-03-16 16:54:28|               16/03/1974|
+----------+----+---+-------+------------+-------------+-------------------+-------------------------+
only showing top 3 rows


- Investigando tipo de duplicidade (as linhas duplicadas são idênticas ou se o mesmo ID possui informações cadastrais diferentes)

In [44]:
# Verifica se o mesmo ID tem dados diferentes nas outras colunas
duplicados_detalhes = clientes.groupBy("id_cliente").count().filter("count > 1")
clientes.join(duplicados_detalhes, "id_cliente", "inner").show()

+----------+----+---+-------+--------------+-------------+-------------------+-------------------------+-----+
|id_cliente|sexo| uf|  renda|        cidade|         nome|    data_nascimento|data_nascimento_formatada|count|
+----------+----+---+-------+--------------+-------------+-------------------+-------------------------+-----+
|      1218|   M| SC|15961.0|     Joinville|Cliente_01218|1974-10-03 09:29:56|               03/10/1974|    2|
|      1934|   F| DF| 4488.0|      Brasília|Cliente_01934|1994-03-06 23:05:59|               06/03/1994|    2|
|      1990|   F| BA|21974.0|      Salvador|Cliente_01990|1996-07-29 22:17:01|               29/07/1996|    2|
|      2042|   F| GO|11395.0|       Goiânia|Cliente_02042|1992-04-24 21:50:02|               24/04/1992|    2|
|      2419|   F| DF| 6214.0|      Brasília|Cliente_02419|1996-07-12 23:14:32|               12/07/1996|    2|
|      3851|   M| SP|15892.0|      Campinas|Cliente_03851|1961-04-11 00:06:01|               11/04/1961|    2|
|

- Tratando os duplicados, mantendo o cadastro mais recente

In [45]:
# Define a janela: agrupa por ID e ordena pela data de atualização mais recente
janela_cliente = Window.partitionBy("id_cliente").orderBy(F.col("data_nascimento").desc())

# Mantém apenas o registro mais recente de cada ID
clientes_unicos = (clientes
                   .withColumn("row_num", F.row_number().over(janela_cliente))
                   .filter("row_num = 1")
                   .drop("row_num"))

In [55]:
# Validando tratamento de dupicidade dos dados
total = clientes_unicos.count()
unicos = clientes_unicos.select("id_cliente").distinct().count()
duplicados = total - unicos
print(f"Total de registros duplicados: {duplicados}")

Total de registros duplicados: 0


- Investigando renda Nula
  
Omissão de renda pode indicar fraude por conta de bots, ou perfis de alto risco que evitam declarar renda real. Investigar se pode ser por falta de uso da conta.

In [58]:
# Investigação: Clientes com renda nula estão transacionando valores altos?
clientes_unicos.filter(F.col("renda").isNull()).summary().show()

+-------+-----------------+----+----+-----+--------------------+-------------+-------------------+-------------------------+
|summary|       id_cliente|sexo|  uf|renda|              cidade|         nome|    data_nascimento|data_nascimento_formatada|
+-------+-----------------+----+----+-----+--------------------+-------------+-------------------+-------------------------+
|  count|               80|  80|  80|    0|                  80|           80|                 80|                       80|
|   mean|        4829.2375|NULL|NULL| NULL|                NULL|         NULL|               NULL|                     NULL|
| stddev|3152.600607189145|NULL|NULL| NULL|                NULL|         NULL|               NULL|                     NULL|
|    min|             1006|   F|  BA| NULL|            Anápolis|Cliente_00163|1955-10-19 02:49:42|               01/01/1963|
|    25%|           1932.0|NULL|NULL| NULL|                NULL|         NULL|               NULL|                     NULL|


#### Contas

In [59]:
print('Qtd. registros:', contas.count(), '| Quantidade de colunas: ', len(contas.columns))
contas.printSchema()

Qtd. registros: 10000 | Quantidade de colunas:  5
root
 |-- id_conta: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- tipo_conta: string (nullable = true)
 |-- status: string (nullable = true)
 |-- data_abertura: string (nullable = true)



- Padronizando os tipos de variáveis

In [60]:
contas = contas.withColumn("data_abertura_formatada", date_format("data_abertura", "dd/MM/yyyy"))
contas.show(2)

- Existe alguma conta sem cliente?

In [61]:
# Retorna apenas as linhas de 'contas' cujo id_cliente NÃO existe em 'clientes'
contas_sem_cliente = contas.join(clientes, on="id_cliente", how="left_anti")

# Conta a volumetria
total_contas_fantasma = contas_sem_cliente.count()
print(f"Quantidade de contas sem cliente: {total_contas_fantasma}")

# Caso existisse, buscaria os exemplos suspeitos para investigação:
# if total_contas_fantasma > 0:
#     contas_sem_cliente.select("id_conta", "id_cliente", "data_abertura").show(10)

Quantidade de contas sem cliente: 0


- Existe cliente sem conta?

In [62]:
# Retorna apenas as linhas de 'clientes' cujo id_cliente NÃO existe em 'contas'
clientes_sem_conta = clientes.join(contas, on="id_cliente", how="left_anti")

# Conta a volumetria
total_clientes_sem_conta = clientes_sem_conta.count()
print(f"Quantidade de clientes sem conta: {total_clientes_sem_conta}")

#  Caso existisse, buscaria os exemplos para o time de Prevenção analisar
# if total_clientes_sem_conta > 0:
#     clientes_sem_conta.select("id_cliente", "nome", "data_cadastro").show(10)

Quantidade de clientes sem conta: 0


#### pix

In [26]:
print('Qtd. registros:', pix.count(), '| Quantidade de colunas: ', len(pix.columns))
pix.printSchema()

Qtd. registros: 120000 | Quantidade de colunas:  5
root
 |-- id_pix: string (nullable = true)
 |-- id_conta: string (nullable = true)
 |-- valor: string (nullable = true)
 |-- status: string (nullable = true)
 |-- data_hora: string (nullable = true)



- Padronizando os tipos de variáveis

In [28]:
pix = pix.withColumn("valor", F.col("valor").cast("float"))
pix = pix.withColumn("data_formatada", date_format("data_hora", "dd/MM/yyyy"))

- Filtando pix com status concluido

In [27]:
pix.groupBy('status').count().show()

[Stage 32:=============================>                            (1 + 1) / 2]

+---------+------+
|   status| count|
+---------+------+
|Concluido|111641|
|Cancelado|  5978|
|   Falhou|  2381|
+---------+------+



In [39]:
pix = pix.filter(F.col("status") == 'Concluido')
pix.groupBy('status').count().show()

+---------+------+
|   status| count|
+---------+------+
|Concluido|111641|
+---------+------+



#### Cartão

In [3]:
print('Qtd. registros:', transacoes.count(), '| Quantidade de colunas: ', len(transacoes.columns))
transacoes.printSchema()

[Stage 5:=============================>                             (1 + 1) / 2]

Qtd. registros: 90000 | Quantidade de colunas:  5
root
 |-- id_compra: string (nullable = true)
 |-- id_conta: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- valor: string (nullable = true)
 |-- data_hora: string (nullable = true)



- Padronizando os tipos de variáveis

In [18]:
transacoes = transacoes.withColumn("data_formatada", date_format("data_hora", "dd/MM/yyyy"))
transacoes = transacoes.withColumn("valor", F.col("valor").cast("float"))
transacoes.show(3)

+---------+--------+-----------+------+-------------------+--------------+
|id_compra|id_conta|  categoria| valor|          data_hora|data_formatada|
+---------+--------+-----------+------+-------------------+--------------+
|        1|  105022|     Outros|348.09|2026-05-15 06:25:06|    15/05/2026|
|        2|  108193|     Viagem|222.75|2026-05-16 02:41:36|    16/05/2026|
|        3|  106290|Combustivel|170.01|2026-06-01 00:53:09|    01/06/2026|
|        4|  106641|     Viagem|179.67|2026-02-15 17:55:52|    15/02/2026|
|        5|  109431|  Streaming|164.73|2026-03-26 17:36:09|    26/03/2026|
|        6|  107637|Restaurante|374.86|2026-05-14 19:53:08|    14/05/2026|
|        7|  107931|Restaurante| 302.5|2026-04-06 18:31:37|    06/04/2026|
|        8|  100741|    Mercado|205.74|2026-04-18 11:26:07|    18/04/2026|
|        9|  102935|     Viagem|126.62|2026-04-19 02:49:17|    19/04/2026|
|       10|  103210|   Farmacia|198.97|2026-04-18 19:32:48|    18/04/2026|
|       11|  105561|    M

- Verificando se existe compra com valor menor ou igual a zero

In [23]:
transacoes.filter(F.col('valor') <= 0.0).count()

0

- Verificando se existe categoria inválida

In [24]:
transacoes.groupBy('categoria').count().show()

[Stage 26:=============================>                            (1 + 1) / 2]

+-----------+-----+
|  categoria|count|
+-----------+-----+
|Combustivel|12849|
|     Outros|12637|
|    Mercado|12942|
|  Streaming|12935|
|Restaurante|13008|
|     Viagem|12701|
|   Farmacia|12928|
+-----------+-----+



- Veirifcando se existe conta inexistente

### Escrita

In [63]:
WRITE_PATH = '/home/tatiane/repositorios/data-projects/02_data_analysis/etl_banco_digital/silver'

In [64]:
clientes_unicos.write.parquet(f"{WRITE_PATH}/clientes")

In [65]:
pix.write.parquet(f"{WRITE_PATH}/pix")

In [66]:
contas.write.parquet(f"{WRITE_PATH}/contas")

In [67]:
transacoes.write.parquet(f"{WRITE_PATH}/transacoes")

In [ ]:
# Exemplo de escrita em csv
# (df_tratado.coalesce(1).write
#  .mode("overwrite")
#  .option("header", "true")
#  .option("sep", ";")
#  .csv("caminho/do/diretorio/base_clientes_unico"))